In [ ]:
import numpy as np
from scipy.special import i0, expit
from scipy.stats import vonmises
from scipy.optimize import minimize, dual_annealing
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.compose import TransformedTargetRegressor

import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

from src.model import jammalamadaka_sharma_corr, angular_absolute_error, CircularTransformer

meta = pd.read_parquet("phenocam_site_meta.parquet")
pheno = pd.read_parquet("phenocam_phenology_model.parquet")
climate = pd.read_parquet("phenocam_site_in_usa_climate.parquet")

In [ ]:
# Construct long-term averages of mean annual temperature, total precipitation, and 3-year anomalies for each
lt_avg_window = 20 # years
anom_avg_window = 3 # years

climate_annual_avg = (
    climate
    .reset_index()
    .groupby(["site", "year"])
    .agg(dict(
        prcp=("sum"),
        tavg=("mean")
    ))
)

climate_annual_avg[f"tavg_{lt_avg_window}yr"] = climate_annual_avg["tavg"].rolling(lt_avg_window).mean()
climate_annual_avg[f"prcp_{lt_avg_window}yr"] = climate_annual_avg["prcp"].rolling(lt_avg_window).mean()
climate_annual_avg[f"tavg_{anom_avg_window}yr"] = climate_annual_avg["tavg"].rolling(anom_avg_window).mean()
climate_annual_avg[f"prcp_{anom_avg_window}yr"] = climate_annual_avg["prcp"].rolling(anom_avg_window).mean()

climate_annual_avg["tavg_anom"] = (climate_annual_avg[f"tavg_{anom_avg_window}yr"] - climate_annual_avg[f"tavg_{lt_avg_window}yr"]) / climate_annual_avg["tavg"].std()
climate_annual_avg["prcp_anom"] = (climate_annual_avg[f"prcp_{anom_avg_window}yr"] - climate_annual_avg[f"prcp_{lt_avg_window}yr"]) / climate_annual_avg["prcp"].std()

In [ ]:
site = "yakimabasin2"
climate_annual_avg.loc[site][["tavg_20yr", "tavg_3yr", "tavg_anom"]].plot()
plt.title(site)
plt.grid()
plt.show()

In [ ]:
# Join phenology data
pheno_subset = pheno.query('direction == "rising"')[["site", "transition_year", "veg_type", "doy_transition_50", "transition_50_uncrt_days"]].copy()

model_data = climate_annual_avg.merge(
    pheno_subset,
    left_on=["site", "year"],
    right_on=["site", "transition_year"],
    how="inner"
)

In [ ]:
# Model fitting
m_lr = TransformedTargetRegressor(
    regressor=LinearRegression(),
    transformer=CircularTransformer(cycle_length=365),
    check_inverse=False
)
m_qr = TransformedTargetRegressor(
    regressor=QuantileRegressor(),
    transformer=CircularTransformer(cycle_length=365),
    check_inverse=False
)

In [ ]:
X = model_data[["tavg_20yr", "tavg_anom", "prcp_20yr", "prcp_anom"]]
y = model_data["doy_transition_50"]

m_lr.fit(X, y)
m_qr.fit(X, y)

In [ ]:
y_hat_lr = m_lr.predict(X)
y_hat_qr = m_qr.predict(X)

In [ ]:
plt.scatter(y, y_hat_lr, s=4, label="LinReg")
plt.scatter(y, y_hat_qr, s=4, label="QuantReg")
plt.gca().set_aspect("equal")
plt.xlim(0, 365)
plt.ylim(0, 365)
plt.axline((0, 0), (1, 1), color="black")
plt.grid()
plt.legend()
plt.title("Global model")
plt.xlabel("True SOS")
plt.ylabel("Model SOS")
plt.show()

In [ ]:
models_by_veg = {}

for veg, data in model_data.groupby("veg_type"):
    this_model = TransformedTargetRegressor(
        regressor=LinearRegression(),
        transformer=CircularTransformer(cycle_length=365),
        check_inverse=False
    )

    X = data[["tavg_20yr", "tavg_anom", "prcp_20yr", "prcp_anom"]]
    y = data["doy_transition_50"]

    this_model.fit(X, y)

    models_by_veg[veg] = this_model

In [ ]:
def predict_by_group(df, model_dict, feature_cols):
    predictions = []
    
    for group_name, group_data in df.groupby('veg_type'):
        if group_name in model_dict:
            X_group = group_data[feature_cols]
            preds = model_dict[group_name].predict(X_group)
            
            # Keep track of original indices to align predictions correctly
            group_preds = pd.Series(preds, index=group_data.index)
            predictions.append(group_preds)
            
    return pd.concat(predictions).sort_index()

model_data['doy_transition_50_model'] = predict_by_group(model_data, models_by_veg, ["tavg_20yr", "tavg_anom", "prcp_20yr", "prcp_anom"])

In [ ]:
model_data_subset = model_data[model_data.veg_type.isin(["DB", "EN", "GR", "SH"])]

for veg, data in model_data_subset.groupby("veg_type"):
    plt.scatter(data["doy_transition_50"], data["doy_transition_50_model"], s=4, label=veg)
    
plt.gca().set_aspect("equal")
plt.xlim(0, 365)
plt.ylim(0, 365)
plt.axline((0, 0), (1, 1), color="black")
plt.legend()
plt.grid()
plt.title("Model by veg type")
plt.xlabel("True SOS")
plt.ylabel("Model SOS")
plt.show()